In [ ]:
import numpy as np
import pandas as pd
import os
import re
import pickle
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import knn_graph

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import AGNNConv, TransformerConv, global_mean_pool

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from tqdm import tqdm
import wandb

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
# Dataset settings
dataset = "isic"
seg = 8

loaded_file = f"{dataset}_{seg}"
pkl_path = f"slic/resnet/{dataset}/{dataset}_slic_region_features_{seg}.pkl"
print(f"Loading features from {pkl_path} ...")

with open(pkl_path, "rb") as f:
    data_dict = pickle.load(f)

print(f"Loaded {len(data_dict)} image feature sets.")

fname = list(data_dict.keys())[5]
content = data_dict[fname]

print(f"Filename: {content['filename']}")
print(f"Label: {content['label']}")
print(f"Region features shape: {content['region_features'].shape}")
print(f"Segments shape: {content['segments'].shape}")

In [ ]:
# Load metadata
metadata_path = f"../R-GNN/skin-datasets/{dataset}_metadata.csv"
metadata = pd.read_csv(metadata_path, sep=None, engine="python")

# Normalize metadata column names
metadata.columns = metadata.columns.str.lower()

# Drop label columns if present
class_labels = ['lesion_id', 'patient_id', 'label', 'benign_malignant', 'diagnosis_1', 'diagnosis', 'diagnosis_2', 'diagnosis_3'] 
metadata = metadata.drop(columns=[c for c in class_labels if c in metadata.columns], errors='ignore')

# Rename img_id to isic_id if needed
columns_to_rename = {
    'img_id': 'isic_id',
}
metadata.rename(columns={k: v for k, v in columns_to_rename.items() if k in metadata.columns}, inplace=True)

# Normalize IDs
if 'isic_id' not in metadata.columns:
    raise ValueError("Metadata must contain an 'isic_id' column")

metadata['isic_id'] = (
    metadata['isic_id']
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace('.jpg', '', regex=False)
    .str.replace('.png', '', regex=False)
)

# Normalize pkl keys
pkl_keys = [k.split('.')[0].strip().lower() for k in data_dict.keys()]

common_ids = set(pkl_keys) & set(metadata['isic_id'])
print(f"Matched {len(common_ids)} IDs with available metadata")

metadata = metadata[metadata['isic_id'].isin(common_ids)].reset_index(drop=True)
print(f"Matched {len(metadata)} images with available metadata")

# Handle missing values
numeric_cols = metadata.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = metadata.select_dtypes(exclude=[np.number, 'bool']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c not in ['isic_id']]

metadata[numeric_cols] = metadata[numeric_cols].fillna(metadata[numeric_cols].mean())

# Z-standardize numeric and one hot encode categorical
ct = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

meta_features = ct.fit_transform(metadata.drop(columns=['isic_id']))
meta_feature_names = list(ct.get_feature_names_out())

metadata_processed = pd.DataFrame(meta_features, columns=meta_feature_names)
metadata_processed['isic_id'] = metadata['isic_id'].values

print(metadata_processed.shape)

In [ ]:
# Train val test split by image ids
all_ids = metadata_processed['isic_id'].values

train_ids, test_ids = train_test_split(all_ids, test_size=0.2, random_state=42)
train_ids, val_ids = train_test_split(train_ids, test_size=0.1, random_state=42)

print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")

In [ ]:
def remap_segments_to_contiguous(segments: torch.Tensor):
    flat = segments.view(-1).long()
    uniq = torch.unique(flat)
    
    new_ids = torch.arange(uniq.numel(), device=segments.device, dtype=torch.long)
    remapped = torch.empty_like(flat)
    
    for old, new in zip(uniq.tolist(), new_ids.tolist()):
        remapped[flat == old] = new
    remapped = remapped.view_as(segments)
    
    return remapped, int(uniq.numel())

In [ ]:
k= 6
def knn_graph_manual(pos: torch.Tensor, k: int) -> torch.Tensor:
    """
    pos: [N, 2]
    returns edge_index: [2, E] directed edges i->j for k nearest neighbors
    """
    # print("Input pos shape:", pos.shape)
    
    N = pos.size(0)
    
    if N <= 1:
        return torch.empty((2, 0), dtype=torch.long, device=pos.device)

    dist = torch.cdist(pos, pos, p=2)
    dist.fill_diagonal_(float("inf"))  # Remove self-connections

    k_eff = min(k, N - 1)
    nn_idx = dist.topk(k=k_eff, largest=False).indices  # [N, k_eff]

    src = torch.arange(N, device=pos.device).view(-1, 1).repeat(1, k_eff)
    dst = nn_idx
    
    edge_index = torch.stack([src.reshape(-1), dst.reshape(-1)], dim=0)
    
    # make undirected
    edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
    edge_index = torch.unique(edge_index, dim=1)
    
    # print("Output edge_index shape:", edge_index.shape)
    # print("Output edge_index value:", edge_index)
    # print("k=",k)
    # print("k_eff=",k_eff)
    
    return edge_index

In [ ]:
def superpixel_centroids_from_segments(segments: torch.Tensor, num_nodes: int):
    """
    segments: [H, W] with labels in {0..num_nodes-1}
    returns pos: [num_nodes, 2] = [cx, cy] computed from bbox min/max per region
    """
    segments = segments.long()
    H, W = segments.shape
    device = segments.device

    pos = torch.zeros((num_nodes, 2), dtype=torch.float32, device=device)

    for r in range(num_nodes):
        ys, xs = torch.where(segments == r)

        if ys.numel() == 0:
            pos[r] = torch.tensor([0.0, 0.0], device=device)
            continue

        xmin = xs.min().float()
        xmax = xs.max().float()
        ymin = ys.min().float()
        ymax = ys.max().float()

        cx = (xmin + xmax) * 0.5
        cy = (ymin + ymax) * 0.5
        pos[r, 0] = cx
        pos[r, 1] = cy

    return pos

In [ ]:
def normalize_pos_by_image(pos: torch.Tensor, H: int, W: int):
    diag = math.sqrt((H - 1) * (H - 1) + (W - 1) * (W - 1))
    return pos / max(diag, 1e-8)

def edge_geom_from_pos(pos: torch.Tensor, edge_index: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """
    pos: [N, 2] region centers
    edge_index: [2, E]
    returns edge_geom: [E, 2] = [dist, angle]
    dist is Euclidean distance
    angle is atan2(dy, dx) in radians
    """
    
    src, dst = edge_index[0], edge_index[1]
    
    delta = pos[dst] - pos[src]  # [E, 2]
    
    dx = delta[:, 0]
    dy = delta[:, 1]

    dist = torch.sqrt(dx * dx + dy * dy + eps)     # [E] distance
    angle = torch.atan2(dy, dx)                    # [E] in [-pi, pi] angle
    
    edge_geom = torch.stack([dist, angle], dim=1)  # [E, 2]
    
    return edge_geom

def normalize_edge_geom(edge_geom: torch.Tensor) -> torch.Tensor:
    """
    Recommended normalization:
    dist already becomes stable if pos was normalized by diagonal
    angle normalized to [-1, 1]
    """
    out = edge_geom.clone()
    out[:, 1] = out[:, 1] / math.pi
    
    return out

In [ ]:
def build_graph(fname, f_data, meta_proj, metadata_processed, k=k):
    """
    Returns Data with:
      - region nodes + 1 metadata node
      - region-region edges from kNN on region centers
      - meta-region edges fully connected (both directions)
      - edge_geom for ALL edges: [dist, angle] (then your model will MLP -> edge_attr)
    """
    fname_norm = fname.lower().replace(".jpg", "").replace(".png", "").strip()

    row = metadata_processed[metadata_processed["isic_id"] == fname_norm]
    
    if row.empty:
        return None

    # ----- region features -----
    features = f_data["region_features"]
    
    if not isinstance(features, torch.Tensor):
        features = torch.tensor(features, dtype=torch.float32)
    else:
        features = features.float()

    # ----- segments -----
    segments = f_data["segments"]
    
    if not isinstance(segments, torch.Tensor):
        segments = torch.as_tensor(segments)
    segments = segments.long()

    label = int(f_data["label"])

    segments_remap, num_nodes_from_segments = remap_segments_to_contiguous(segments)
    
    num_region_nodes = features.shape[0]
    
    if num_nodes_from_segments != num_region_nodes:
        print(f"Warning: segments nodes {num_nodes_from_segments} != features nodes {num_region_nodes} for {fname_norm}")
        return None

    H, W = segments_remap.shape

    # ----- Region center: bbox min/max -----
    pos_region = superpixel_centroids_from_segments(segments_remap, num_nodes=num_region_nodes)

    pos_region = normalize_pos_by_image(pos_region, H=H, W=W)

    # ----- region-region edges (kNN) -----
    k_eff = min(k, max(num_region_nodes - 1, 1))
    
    edge_index_rr = knn_graph_manual(pos_region, k=k_eff)  # [2, E_rr]

    if edge_index_rr.numel() == 0:
        return None

    edge_geom_rr = edge_geom_from_pos(pos_region, edge_index_rr)
    edge_geom_rr = normalize_edge_geom(edge_geom_rr)       # [E_rr, 2]

    # ----- metadata node feature (project to CNN feature dim) -----
    meta_vals = row.drop(columns=["isic_id"]).values  # [1, M]
    meta_node = torch.tensor(meta_vals, dtype=torch.float32).to(device)  # [1, M]
    
    with torch.no_grad():
        meta_node = meta_proj(meta_node)  # [1, F] to match region feature dim

    # ----- combine nodes -----
    x_all = torch.cat([features, meta_node], dim=0)         # [N+1, F]
    meta_idx = x_all.size(0) - 1

    # metadata node position at image center (normalized)
    center_xy = torch.tensor([0.5, 0.5], dtype=torch.float32) 
    
    # mean of region positions
    center_xy = pos_region.mean(dim=0)

    pos_meta = center_xy.view(1, 2)
    pos_all = torch.cat([pos_region, pos_meta], dim=0)      # [N+1, 2]

    # meta region edges
    region_idx = torch.arange(num_region_nodes, dtype=torch.long)

    meta_to_region = torch.stack([
        torch.full((num_region_nodes,), meta_idx, dtype=torch.long),
        region_idx
    ], dim=0)

    edge_index_meta = meta_to_region  # One directional
    region_to_meta = meta_to_region.flip(0) # Both direction

    # shift indices to same device
    meta_to_region = meta_to_region.to(edge_index_rr.device)
    region_to_meta = region_to_meta.to(edge_index_rr.device)
    region_idx = region_idx.to(edge_index_rr.device)

    edge_index_meta = torch.cat([meta_to_region, region_to_meta], dim=1)  # [2, 2N]

    # compute geometry for meta edges using pos_all
    edge_geom_meta = edge_geom_from_pos(pos_all, edge_index_meta)
    edge_geom_meta = normalize_edge_geom(edge_geom_meta)  # [2N, 2]

    # ----- final edge_index and edge_geom -----
    edge_index = torch.cat([edge_index_rr, edge_index_meta], dim=1)
    edge_geom = torch.cat([edge_geom_rr, edge_geom_meta], dim=0)

    y = torch.tensor([label], dtype=torch.long)

    graph = Data(x=x_all, pos=pos_all, edge_index=edge_index, edge_geom=edge_geom, y=y)
    graph.isic_id = fname_norm
    
    return graph

In [ ]:
cnn_feature_dim = content['region_features'].shape[1]
meta_feature_dim = metadata_processed.shape[1] - 1

meta_proj = nn.Linear(meta_feature_dim, cnn_feature_dim).to(device)

train_graphs, val_graphs, test_graphs = [], [], []

for fname, f_data in data_dict.items():
    graph = build_graph(fname, f_data, meta_proj, metadata_processed, k=k)
    
    if graph is None:
        continue

    if graph.isic_id in train_ids:
        train_graphs.append(graph)
    elif graph.isic_id in val_ids:
        val_graphs.append(graph)
    elif graph.isic_id in test_ids:
        test_graphs.append(graph)

print(f"Built {len(train_graphs)} train, {len(val_graphs)} val, and {len(test_graphs)} test graphs.")

train_ids_set = set([g.isic_id for g in train_graphs])
val_ids_set = set([g.isic_id for g in val_graphs])
test_ids_set = set([g.isic_id for g in test_graphs])

print("Train–Val overlap:", len(train_ids_set & val_ids_set))
print("Train–Test overlap:", len(train_ids_set & test_ids_set))
print("Val–Test overlap:", len(val_ids_set & test_ids_set))

In [ ]:
batch_size = 32

train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=batch_size, shuffle=False)

In [ ]:
class EdgeMLP(nn.Module):
    def __init__(self, in_dim=2, hidden=32, out_dim=2, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, edge_geom):
        return self.net(edge_geom) 

class HybridTransformerAGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels=512, num_classes=2, heads=2, dropout=0.3):
        super().__init__()
        self.dropout = dropout

        self.edge_mlp = EdgeMLP(in_dim=2, hidden=32, out_dim=2, dropout=dropout)

        self.transformer1 = TransformerConv(
            in_channels=in_channels,
            out_channels=hidden_channels,
            heads=heads,
            edge_dim=2
        )
        self.bn1 = nn.BatchNorm1d(hidden_channels)

        self.agnn1 = AGNNConv(requires_grad=True)
        self.bn2 = nn.BatchNorm1d(hidden_channels)

        self.lin1 = nn.Linear(hidden_channels, 128)
        self.bn_fc1 = nn.BatchNorm1d(128)
        
        self.lin2 = nn.Linear(128, num_classes)

    def forward(self, x, edge_index, edge_geom, batch, return_embedding=False):
        edge_attr = self.edge_mlp(edge_geom)

        x = self.transformer1(x, edge_index, edge_attr=edge_attr)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.agnn1(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        graph_embed = global_mean_pool(x, batch)

        if return_embedding:
            return graph_embed

        x = self.lin1(graph_embed)
        x = self.bn_fc1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        x = self.lin2(x)
        
        return x

In [ ]:
wandb.init(
    project="R-GNN",
    name=f"{loaded_file}-meta-edge",
    config={
        "epochs": 50,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "model": "GeoMeta"
    }
)

In [ ]:
model = HybridTransformerAGNN(in_channels=cnn_feature_dim, heads=heads).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=wandb.config["lr"],
    weight_decay=wandb.config["weight_decay"]
)

best_val_acc = 0.0

save_dir = "best-models/"
os.makedirs(save_dir, exist_ok=True)

def get_highest_saved_acc(save_dir, loaded_file):
    accs = []
    for f in os.listdir(save_dir):
        match = re.search(rf"{re.escape(loaded_file)}-acc-(\d+\.\d+)", f)
        if match:
            accs.append(float(match.group(1)))
    return max(accs) if accs else 0.0

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=6
)

for epoch in range(1, wandb.config["epochs"] + 1):
    model.train()
    total_loss = 0.0
    correct_train = 0
    total_train = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch} [Training]"):
        batch = batch.to(device)
        optimizer.zero_grad()

        out = model(batch.x, batch.edge_index, batch.edge_geom, batch.batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        preds = out.argmax(dim=1)
        correct_train += (preds == batch.y).sum().item()
        total_train += batch.y.size(0)

    train_loss = total_loss / max(len(train_loader), 1)
    train_acc = correct_train / max(total_train, 1)

    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.edge_geom, batch.batch)
            val_loss += criterion(out, batch.y).item()

            preds = out.argmax(dim=1)
            correct += (preds == batch.y).sum().item()
            total += batch.y.size(0)

    val_loss = val_loss / max(len(val_loader), 1)
    val_acc = correct / max(total, 1)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    scheduler.step(val_loss)

    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_acc": train_acc,
        "val_acc": val_acc,
    })

    current_best_saved_acc = get_highest_saved_acc(save_dir, loaded_file)

    if val_acc > current_best_saved_acc:
        best_val_acc = val_acc

        for f in os.listdir(save_dir):
            if f.startswith(f"best_model_{loaded_file}"):
                os.remove(os.path.join(save_dir, f))

        save_path = os.path.join(
            save_dir,
            f"best_gnn_model_{loaded_file}-acc-{best_val_acc:.2f}-{heads}.pth"
        )
        torch.save(model.state_dict(), save_path)
        print(f"Best model saved at epoch {epoch} with Val Acc: {best_val_acc:.2f}")
    else:
        best_val_acc = current_best_saved_acc
        print(f"Not saved (Current best in folder: {current_best_saved_acc:.2f})")

In [ ]:
print("\nEvaluating on Test Set...")

model_path = f"/home/user/R-GNN/best_models/best_gnn_model_{loaded_file}-acc-{best_val_acc:.2f}-{heads}.pth"
model.load_state_dict(torch.load(model_path, map_location=device))
print(f"Loaded Model: {model_path}")

model.eval()

test_loss = 0.0
correct = 0
total = 0

y_true, y_pred = [], []
embeddings = []
labels_list = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.edge_geom, batch.batch)
        
        graph_embed = model(batch.x, batch.edge_index, batch.edge_geom, batch.batch, return_embedding=True)
        loss = criterion(out, batch.y)
        test_loss += loss.item()

        preds = out.argmax(dim=1)
        correct += (preds == batch.y).sum().item()
        total += batch.y.size(0)

        y_true.extend(batch.y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

        embeddings.append(graph_embed.cpu().numpy())
        labels_list.append(batch.y.cpu().numpy())

test_loss = test_loss / max(len(test_loader), 1)
test_acc = correct / max(total, 1)

print(f"\nTest Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")

wandb.log({
    "test_loss": test_loss,
    "test_acc": test_acc
})

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["benign", "malignant"]))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

In [ ]:
wandb.finish()